# AIHub 증강 학습 + 모델 비교 (run_experiment_augmented)

검증된 baseline(Kaggle train, LB 0.916) 위에 **AIHub 조합 1,3을 추가 학습 데이터로** 사용해
YOLO와 torchvision 계열(SSD300 / Faster R-CNN / RetinaNet / FCOS)을 **동일한 데이터**로
학습해 비교한다.

원칙:
1. **val은 Kaggle 대표 검증셋으로 유지** — AIHub는 train에만 추가.
2. **누수 0(그룹 확장 버전)** — 같은 조합(combo) 문자열뿐 아니라, 지각 해시로 확인한
   **근접중복 장면**(같은 트레이/배경에서 약 1개만 바꿔 찍은 사진)까지 하나의 그룹으로
   묶어 train/val이 절대 갈리지 않게 한다(`build_leakage_safe_groups`). K-Fold 감사에서
   조합 문자열 기준 분할만으로는 val의 ~30%가 이런 근접중복이었던 게 확인됐다.
3. **실제 Kaggle 테스트셋 대비 leakage도 사전 제거** — AIHub 조합 이미지 중 실제
   테스트셋과 근접중복인 것은 `audit_leakage.audit()`로 찾아 학습에서 아예 제외한다
   (감사에서 6장 발견됨).
4. **혼합 이미지 박스 보존** — 56외 약 박스도 그대로 학습(지우면 라벨 없는 알약이 생겨 악영향). 56이 하나도 없는 이미지만 제외.
5. **category_id는 K-코드로 통일** — AIHub/Kaggle 동일 체계.
6. **모델마다 실행 로직은 동일** — 아래에서 한 번만 만든 `train_aug`/`val_aug`를 YOLO와
   torchvision 모델 전부가 그대로 재사용한다.
7. **제출은 56 클래스로 제한** — 그 외 예측은 자동 제거(현재 `src/inference.py`는 YOLO 전용이라 제출은 YOLO 기준).

> **exp011 대비 변경점**: exp011은 콤보 문자열(`combo_group_key`)만으로 train/val을
> 나눠 mAP@75:95 0.9914를 얻었지만, 이후 K-Fold 감사에서 val의 ~30%가 "조합은 다르지만
> 사실상 같은 장면"인 근접중복으로 드러났다. 이 노트북은 그 문제를 고친 **exp016**을
> 새 이름으로 만든다(exp011은 "정리 전" 비교 기준으로 그대로 둔다 — 점수 차이가
> 곧 근접중복 때문에 부풀려졌던 정도의 추정치가 된다).

> monorepo(`beamsearch/CJY`)로 옮겨오며 `configs/joelchoi`, `data/joelchoi`,
> `experiments/joelchoi` 개인 하위 폴더는 정리(삭제)했다. 아래 경로는 모두
> 개인 폴더(`CJY`)의 하위(`configs/experiment`, `data/`, `experiments/`, `submissions/`)를
> 가리킨다.

In [ ]:
import os

# RF-DETR(Multi-Scale Deformable Attention)의 grid_sample backward가 아직
# PyTorch MPS 백엔드에 없어서(forward만 됨) 이 연산만 자동으로 CPU 폴백하게
# 켜둔다. torch를 이 노트북 어디서든 import하기 전에 설정해야 가장 안전하므로
# 첫 셀 맨 위에 둔다(src/train_hf.py에도 동일하게 방어적으로 넣어뒀지만,
# 노트북에서 다른 셀이 먼저 torch를 import하는 경우를 대비한 이중 안전장치).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from pathlib import Path

from src.audit_leakage import audit
from src.data.aihub_converter import convert_aihub_to_coco
from src.data.coco_to_yolo import prepare_yolo_dataset
from src.data.kaggle_converter import convert_kaggle_to_coco
from src.data.merge import merge_for_augmentation
from src.data.subset import (
    build_leakage_safe_groups,
    combo_group_key,
    create_subset,
    exclude_images,
)
from src.train import train_torchvision
from src.train_yolo import run_yolo_experiment
from src.utils import load_config

# 공용 폴더 직접 참조(개인 하위 폴더 없음)
CONFIGS_DIR = PROJECT_ROOT / "configs" / "experiment"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
DATA_DIR = PROJECT_ROOT / "data"
YOLO_DIR = DATA_DIR / "yolo_aug_clean"
AIHUB_COMBOS = [1, 3]

print("CONFIGS_DIR:", CONFIGS_DIR)
print("EXPERIMENTS_DIR:", EXPERIMENTS_DIR)
print("YOLO_DIR:", YOLO_DIR)


## 1단계 — Kaggle train → COCO, AIHub 조합 변환

In [ ]:
kaggle_coco = convert_kaggle_to_coco()
allowed_ids = {c["id"] for c in kaggle_coco["categories"]}  # 제출 대상 56
print("Kaggle 클래스:", len(allowed_ids))

aihub_coco_raw = convert_aihub_to_coco(combo_nums=AIHUB_COMBOS)


## 2단계 — 실제 Kaggle 테스트셋 대비 leakage 감사 + AIHub 필터링

`combo_group_key`/`build_leakage_safe_groups`는 우리가 만드는 train/val **내부**
분할만 다룬다. 이 단계는 그것과 별개로, 학습에 넣으려는 AIHub 이미지가 **실제
Kaggle 비공개 테스트셋**과 근접중복인지 확인한다(진짜 컨닝 위험 — 6장 발견됨).
발견된 이미지는 아예 학습에서 제외한다.

In [ ]:
audit_result = audit(combos=AIHUB_COMBOS)
print("\n판정:", audit_result["verdict"])
print(
    "AIHub 근접중복(실제 테스트셋과):",
    len(audit_result["aihub_leak_paths"]),
    "장 → 제외 예정",
)

aihub_coco = exclude_images(aihub_coco_raw, audit_result["aihub_leak_paths"])


## 3단계 — 근접중복 인식 그룹 구성 (누수 안전 그룹)

`combo_group_key`만 쓰면 조합 문자열이 다른 근접중복 장면(같은 트레이에서 약 1개만
교체)이 train/val로 갈릴 수 있다(K-Fold 감사에서 확인). `build_leakage_safe_groups`로
Kaggle 전체 + (필터링된) AIHub 이미지 전체를 대상으로 지각 해시 기반 그룹을 미리
만들어 두고, 아래 4단계의 `create_subset`/`merge_for_augmentation`에 그대로 넘긴다.

주의: 이미지 수(수천 장) 기준 O(n^2) 해시 비교라 수십 초 정도 걸릴 수 있다.

In [ ]:
_pool_images = kaggle_coco["images"] + aihub_coco["images"]
group_overrides = build_leakage_safe_groups({
    "images": _pool_images,
    "annotations": [],
    "categories": [],
})


## 4단계 — train/val 분할 + AIHub 증강 병합 (그룹화 fix 적용)

val은 Kaggle 대표 검증셋으로 유지하되, 이번엔 `group_overrides`를 넘겨 근접중복
장면이 train/val로 갈리지 않도록 한다.

In [ ]:
train_k, val_k = create_subset(
    kaggle_coco, tier="full", test_size=0.2, seed=42, group_overrides=group_overrides
)

train_aug, val_aug = merge_for_augmentation(
    base_train=train_k,
    base_val=val_k,
    extra_coco=aihub_coco,
    allowed_ids=allowed_ids,
    drop_pure_irrelevant=True,
    group_overrides=group_overrides,
)


### 검증 — 근접중복 재점검 (그룹화 fix 전/후 비교)

exp011 때는 이 검사에서 val의 ~30%가 근접중복으로 잡혔다. 그룹화가 제대로 됐다면
이번엔 0에 가까워야 한다(0이 아니면 cutoff나 그룹 병합 로직을 다시 점검할 것).

In [ ]:
from src.audit_leakage import _dhash, _gray_vec, _rmse


def audit_near_duplicates(
    train_coco, val_coco, hash_size=16, hash_cutoff=12, rmse_cutoff=0.06
):
    """train/val 사이 지각 해시 근접 중복(사실상 같은 장면일 가능성)을 스캔."""
    train_paths = [Path(im["file_name"]) for im in train_coco["images"]]
    val_paths = [Path(im["file_name"]) for im in val_coco["images"]]

    train_dh = [(_dhash(p, size=hash_size), p) for p in train_paths]
    train_dh = [(d, p) for d, p in train_dh if d is not None]

    near_dups, affected_val = [], set()
    for vp in val_paths:
        vd = _dhash(vp, size=hash_size)
        if vd is None or not train_dh:
            continue
        best_d, best_p = min(
            ((bin(vd ^ d).count("1"), p) for d, p in train_dh), key=lambda x: x[0]
        )
        if best_d <= hash_cutoff:
            vv, sv = _gray_vec(vp), _gray_vec(best_p)
            if vv is not None and sv is not None and _rmse(vv, sv) <= rmse_cutoff:
                near_dups.append((vp, best_p, best_d, _rmse(vv, sv)))
                affected_val.add(vp)

    pct = 100 * len(affected_val) / max(len(val_paths), 1)
    print(
        f"val {len(val_paths)}장 중 근접중복 의심 {len(affected_val)}장({pct:.1f}%) | "
        f"총 매칭 {len(near_dups)}쌍"
    )
    for vp, tp, hd, r in near_dups[:5]:
        print(f"    VAL {vp.name}  ~~  TRAIN {tp.name}  (hash {hd}, rmse {r:.3f})")
    return near_dups


_ = audit_near_duplicates(train_aug, val_aug)


### 참고 — 중단된 학습 재개하기 (`training.resume`)

RF-DETR처럼 오래 걸리는 학습이 중간에 끊기는 경우(VSCode reload, 메모리 부족 등)를 대비해
세 학습 함수(`train_torchvision`, `train_rfdetr`, `run_yolo_experiment`/`train_yolo`) 모두
**매 epoch마다 체크포인트를 저장**하고, config에 `training.resume: true`를 추가하면
멈춘 지점(epoch)부터 이어서 학습한다.

사용법:
1. 실험 config yaml의 `training:` 아래에 `resume: true`를 추가한다.
2. **같은 `experiment.name`**으로 다시 학습 함수를 실행하면(즉 같은
   `experiments/<exp_name>/` 디렉토리를 가리키면) 자동으로 이어서 진행된다.
3. 체크포인트가 없으면(첫 실행) 경고만 출력하고 처음부터 시작하므로, 항상
   `resume: true`로 둬도 안전하다.

실제 사용 예 — RF-DETR 스모크 테스트 → 본 학습 이어가기:
`exp017_rfdetr_aug_2ep.yaml`(epochs=2, 빠른 동작 확인용)과
`exp018_rfdetr_aug.yaml`(epochs=30, 본 학습)은 **`experiment.name`이 둘 다
`exp017_rfdetr_aug`로 동일**해서 같은 결과 디렉토리를 공유한다. 먼저 2ep로 돌려
에러 없이 끝까지 도는지 확인한 뒤, `exp018_rfdetr_aug.yaml`(`training.resume: true`
설정됨)을 실행하면 2ep 스모크 테스트의 체크포인트에 이어서 나머지 28epoch만 마저
학습한다 — 처음부터 다시 돌리지 않는다.

백엔드별 저장 위치/방식:
- **torchvision** (`train_torchvision`, `src/train.py`) / **RF-DETR** (`train_rfdetr`,
  `src/train_hf.py`): `experiments/<exp_name>/weights/checkpoint.pt`에
  `{epoch, model_state, optimizer_state, scheduler_state, best_map, history}`를 저장.
  freeze_backbone_epochs를 쓰는 모델은 재개 시점이 unfreeze 이후면 자동으로 전체 unfreeze
  상태로 복원된다.
- **YOLO** (`run_yolo_experiment`/`train_yolo`, `src/models/yolo_wrapper.py`):
  ultralytics 자체 resume 메커니즘을 사용 — `experiments/<exp_name>/weights/last.pt`가
  있으면 그 run의 저장된 학습 인자(args.yaml)를 그대로 읽어 이어서 학습한다(다른 인자를
  같이 넘기지 않음, 공식 권장 방식).

이미 목표 epoch까지 끝난 상태에서 `resume: true`로 다시 실행하면 재학습 없이 저장된 결과를
그대로 사용한다(안내 메시지만 출력).

## 5단계 — YOLO 학습 (exp016, exp011의 그룹화 fix 버전)

YOLO용 데이터셋(`data.yaml`)을 한 번 만들어 두면, 아래 6단계의 torchvision 비교
실험들은 같은 `train_aug`/`val_aug` COCO dict를 **그대로 재사용**한다.

In [ ]:
yaml_path = prepare_yolo_dataset(train_aug, val_aug, YOLO_DIR, symlink=True)
CLASS_MAP = YOLO_DIR / "class_map.json"  # 전체 클래스(K-코드) 매핑

yolo_cfg = load_config(CONFIGS_DIR / "exp016_yolo11n_aug_clean.yaml")
yolo_metrics = run_yolo_experiment(yolo_cfg, yaml_path, EXPERIMENTS_DIR)
print(yolo_metrics)


## 6단계 — torchvision + RF-DETR 모델 비교 (SSD300 / Faster R-CNN / RetinaNet / FCOS / RF-DETR)

YOLO와 **완전히 동일한**(그룹화 fix 적용된) `train_aug`/`val_aug`를 그대로 넘긴다.
FCOS(`exp015_fcos_aug.yaml`)는 클래스 수 증가 + 전체 데이터로 메모리가 빠듯해질 수
있어 물리 배치를 줄이고 `accumulate_grad_batches`로 유효 배치를 보존하도록
설정해 뒀다.

Sparse R-CNN/DINO-DETR는 Mac(MPS) 환경에 맞지 않아(각각 Detectron2 전용, CUDA
전용 커스텀 커널 필요) 대신 **RF-DETR**(HuggingFace `transformers`)을 transformer
계열 대표로 추가했다. RF-DETR은 torchvision과 forward 시그니처가 달라
`src/train_hf.py::train_rfdetr`라는 별도 함수로 학습한다.

`COMPARISON_CONFIGS`에 `exp017_rfdetr_aug_2ep.yaml`(epochs=2 스모크 테스트)과
`exp018_rfdetr_aug.yaml`(epochs=30 본 학습, `training.resume: true`)을 순서대로
넣어 뒀다 — 둘은 `experiment.name`이 같아서 2ep 테스트가 에러 없이 통과하면
이어서 exp018이 그 체크포인트부터 나머지 28epoch을 마저 학습한다(바로 위
'중단된 학습 재개하기' 참고). PicklingError(DataLoader collate_fn을 로컬
클로저로 정의해서 num_workers>0일 때 pickle 실패하던 문제)는 `src/train_hf.py`에서
`collate_fn`을 모듈 레벨 클래스로 옮기고 기본 `num_workers`를 0으로 낮춰 수정했다.


In [ ]:
from src.train_hf import train_rfdetr

COMPARISON_CONFIGS = [
    "exp017_rfdetr_aug_2ep.yaml",
    "exp018_rfdetr_aug.yaml",
    "exp012_ssd300_aug.yaml",
    # "exp013_fasterrcnn_aug.yaml",
    "exp014_retinanet_aug.yaml",
    "exp015_fcos_aug.yaml",
]

comparison_metrics = {}
for cfg_name in COMPARISON_CONFIGS:
    cfg = load_config(CONFIGS_DIR / cfg_name)
    exp_name = cfg["experiment"]["name"]
    model_type = cfg["model"]["type"]
    print(f"\n=== {exp_name} 학습 시작 (type={model_type}) ===")

    if model_type == "torchvision":
        metrics = train_torchvision(cfg, train_aug, val_aug, EXPERIMENTS_DIR)
    elif model_type == "huggingface":
        metrics = train_rfdetr(cfg, train_aug, val_aug, EXPERIMENTS_DIR)
    else:
        raise ValueError(f"알 수 없는 model.type: {model_type}")

    comparison_metrics[exp_name] = metrics
    print(f"{exp_name} 완료: mAP@75:95 = {metrics['final_map75_95']:.4f}")


## 7단계 — 결과 비교

exp011(그룹화 fix 전) vs exp016(fix 후)를 나란히 보여주고, torchvision 4종
(exp012~015, fix 후 데이터로 학습됨)도 함께 비교한다.

In [ ]:
import json

import pandas as pd

COMPARE_EXPS = [
    "exp011_yolo11n_aug",  # 참고용: 그룹화 fix 전(근접중복 포함 가능성)
    "exp016_yolo11n_aug_clean",  # 그룹화 fix 후
    "exp012_ssd300_aug",
    # "exp013_fasterrcnn_aug",
    "exp014_retinanet_aug",
    "exp015_fcos_aug",
    "exp017_rfdetr_aug",
]

results = []
for exp_name in COMPARE_EXPS:
    metrics_file = EXPERIMENTS_DIR / exp_name / "metrics.json"
    if not metrics_file.exists():
        print(f"(건너뜀: {exp_name} metrics.json 없음 — 아직 학습 안 됨)")
        continue
    m = json.load(open(metrics_file))
    results.append({
        "experiment": exp_name,
        "model": m.get("model", "unknown"),
        "mAP@75": m.get("map75") or m.get("final_map75", 0),
        "mAP@75:95": m.get("map75_95") or m.get("final_map75_95", 0),
        "epochs": m.get("epochs", 0),
    })

df = pd.DataFrame(results)
if not df.empty:
    df = df.sort_values("mAP@75:95", ascending=False)
df


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Apple SD Gothic Neo"
plt.rcParams["axes.unicode_minus"] = False

if not df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df))
    width = 0.25
    ax.bar(x, df["mAP@75"], width, label="mAP@75")
    ax.bar([i + width for i in x], df["mAP@75:95"], width, label="mAP@75:95")
    ax.set_xticks(x)
    ax.set_xticklabels(df["experiment"], rotation=45, ha="right")
    ax.set_ylabel("mAP")
    ax.set_title("모델별 성능 비교 (exp011=그룹화 fix 전, 나머지=fix 후 동일 데이터)")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 8단계 — 제출용 class_map(56 제한) + 추론 + 제출 (YOLO 기준)
현재 추론 파이프라인(`src/inference.py`)은 YOLO 전용이라 exp016(그룹화 fix 버전)
가중치로 제출 파일을 만든다.

In [ ]:
from src.inference import restrict_class_map, run_inference, save_submission

EXP_NAME = yolo_cfg["experiment"]["name"]
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
SUBMISSION = PROJECT_ROOT / "submissions" / f"{EXP_NAME}_submission.csv"

# 56 클래스만 남긴 제출용 class_map
CLASS_MAP_SUBMIT = restrict_class_map(CLASS_MAP, allowed_ids)

preds = run_inference(
    weights_path=WEIGHTS,
    class_map_path=CLASS_MAP_SUBMIT,
    test_dir=KAGGLE_DATA / "test_images",
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=yolo_cfg["data"]["img_size"],
)
save_submission(preds, SUBMISSION)
print("제출 파일:", SUBMISSION)


## 9단계 — 제출 파일 점검

In [ ]:
import pandas as pd

sub_df = pd.read_csv(SUBMISSION)
test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(sub_df["image_id"].unique())
bad = set(sub_df["category_id"].unique()) - allowed_ids
print(
    f"행 {len(sub_df)}, 예측 이미지 {len(pred_ids)}/{len(test_ids)}, "
    f"고유 category {sub_df['category_id'].nunique()}"
)
print("56 외 category_id?", bad if bad else "없음(정상)")
sub_df.head()
